# **CREACIÓN DEL ENTORNO (.venv)**   
Para que funcione correctamente:
1. Instalamos python la versión 3.13.12 (es la que he probado yo)

2. Creamos un entorno virtual (recomendado para no tener trescientos paquetes instalados)
    - Haciendo Crtl+Shift+P y seleccionando "Python: Create Environment".
    - Te tiene que haber creado una dirección en la carpeta .venv.
    - Podemos comprobarlo abriendo una terminal y ponemos "pip --version" y tiene que salir la dirección de la carpeta .venv.

# **BIBLIOTECAS NECESARIAS**

In [1]:
# Para la extracción de los ids de los videos
import requests
import re
import json
from bs4 import BeautifulSoup
from wonderwords import RandomWord
import random
import datetime
from tqdm.notebook import tqdm
import time


# Para la extracción de videos con la API
from googleapiclient.discovery import build
from yt_dlp import YoutubeDL
import pprint as pprint
import re
import glob
import os

import numpy as np
import pandas as pd
from pandas import DataFrame

from concurrent.futures import ThreadPoolExecutor, as_completed

# **EXTRACCIÓN DE IDS**

Función para devolver 100 ids - los resultados de una query. Con esta función nos aseguramos de que tienen subtítulos y duración media.

In [2]:
def get_video_ids(query):
    '''
    Devuelve una lista de 100 video_ids de YouTube a partir de una consulta de búsqueda
    Estos vídeos cumplen con el filtro de formato de duración media y subtítulos presentes
    '''
    
    url = "https://www.youtube.com/results"
    params = {"search_query": query,
              "sp": "EgQQASgB" #Filtro para videos formato media duración con subtítulos
              }
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    response = requests.get(url, params=params, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    # Buscar el script que contiene ytInitialData
    scripts = soup.find_all("script")

    if not scripts:
        print("got no scripts for query", query)

    for script in scripts:
        if "ytInitialData" in script.text:
            json_text = re.search(r"ytInitialData\s*=\s*(\{.*\});", script.text)
            if json_text:
                data = json.loads(json_text.group(1))
                break
    else:
        print("got no ytInitialData for query", query)
        return []

    # Buscar todos los videoId dentro del JSON
    video_ids = set()
    def extract_ids(obj):
        if isinstance(obj, dict):
            for k, v in obj.items():
                if k == "videoId":
                    video_ids.add(v)
                else:
                    extract_ids(v)
        elif isinstance(obj, list):
            for item in obj:
                extract_ids(item)

    extract_ids(data)

    return list(video_ids)


# 🔹 Ejemplo
# ids = get_video_ids("x after:2026-02-15")
# print(ids)


Función para encontrar ids de videos aleatorios (cada id corresponde a una búsqueda de una palabra aleatoria)

In [3]:
def get_random_ids(num_ids=25, after_date=None, before_date=None):
    '''
    Devuelve una lista de "num_ids" video_ids aleatorios a partir de palabras aleatorias y la función get_video_ids.
    Por cada palabra aleatoria se realiza una búsqueda en Youtube de videos que contengan esa palabra en el título y que hayan sido publicados entre "after_date" y "before_date".
    '''
    ##Habría que añadir algo que controle que la fecha de inicio no sea posterior a la de fin, o que no se introduzcan fechas futuras, etc. 
    # Porque si no se mete en un bucle infinito.
    lista_palabras_aleatorias = []
    lista_ids_aleatorios = []
    w = RandomWord()

    pbar = tqdm(total=num_ids)
    while len(lista_ids_aleatorios) < num_ids:
        try: 
            random_word = w.word()
            query = f'\"{random_word}\" intitle:{random_word}'
            if after_date:
                query += f' after:' + str(after_date)
            if before_date:
                query += f' before:' + str(before_date)
            #print("running query", query)
            lista_ids_aleatorios.append(random.choice(get_video_ids(query)))
            lista_palabras_aleatorias.append(random_word)
            time.sleep(0.2) #para no hacer muchas queries seguidas
            pbar.update(1)
        except Exception as e: pass #print("Ran into exception", e, "for word", random_word) #happens quite often when no videos found for a word in the last day

    pbar.close()
    return lista_palabras_aleatorias, lista_ids_aleatorios

In [4]:
# palabras, ids = get_random_ids(num_ids=5, after_date=str(datetime.date.today()-datetime.timedelta(days=1)))
# #print(ids)
# print(list(zip(palabras,ids)))

In [5]:
#ejemplo de extraccion de informacion con requests para un video. 
# En principio este método de extracción no se va a utilizar, hacemos la extracción con la API para tener acceso a los subtítulos, etc
# url = f"https://www.youtube.com/watch?v={ids[0]}"
# headers = {"User-Agent": "Mozilla/5.0"}

# r = requests.get(url, headers=headers)

# match = re.search(r"ytInitialPlayerResponse\s*=\s*(\{.*?\});", r.text)

# data = json.loads(match.group(1))

# data['videoDetails']

# **GUARDAR API KEY Y CREAR EL "SERVIDOR"**
La función build nos construye un objeto de la API de Youtube, que usaremos para que llame a la API y acceder a los datos.

In [6]:
# Guardamos nuestra API_KEY leyendo de la carpeta privada
with open("./Private/claves.json", "r", encoding="utf-8") as archivo:
    claves = json.load(archivo)
API_KEY = claves["Clave_API"]
if API_KEY:
    print("Loaded API_KEY succesfully")

Loaded API_KEY succesfully


In [7]:
# Construimos el objeto de la API de YouTube
youtube = build('youtube', 'v3', developerKey=API_KEY)

# **FUNCIÓN PARA LIMPIAR EL TEXTO**

In [8]:
#Función temporal para limpiar texto
import re

def clean_vtt(text):
    text = re.sub(r"WEBVTT.*\n", "", text)
    text = re.sub(r"\d+:\d+:\d+\.\d+ --> .*", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\n+", "\n", text)
    return text.strip()

def clean_vtt_smart(text):
    lines = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        if not lines or not line in lines[-1]:
            lines.append(line)
    return " ".join(lines)

# **GENERAMOS UNA PEQUEÑA BASE DE DATOS**
Lo guardaremos en un archivo csv, guardando la información de los 5 videos que hemos encontrado antes.


In [9]:
# Lista de los id de los videos (usando el id el coste de la consulta es el mínimo)
# list_id = ids
# print("ids", list_id)

# # Creo el dataframe vacío
# df_data = pd.DataFrame({
#     "ID": [],
#     "Titulo": [],
#     "Descripcion": [],
#     "Visualizaciones": [],
#     "Numero_likes": [],
#     "Duracion": [],
#     "Fecha_publicacion": [],
#     "Titulo_canal": [],
#     "Subtitulos": []
# })
# df_data

In [10]:
from urllib import response


def get_info(id_video):
    """ Saca la información del video, si hay subtítulos los limpia, y añade dichos datos al dataframe. """
    
    # Inicializamos la variable de los subtitulos
    cleant_sub = None

    # Hacemos la llamada a la API para obtener los detalles del video
    request = youtube.videos().list(
        part="snippet,contentDetails,statistics,status,topicDetails,recordingDetails",
        id=id_video
    )  

    # Ejecutamos la solicitud
    response = request.execute()

    if not response["items"]:
        return None
    
    video = response["items"][0]

    has_captions = video["contentDetails"].get("caption") in ["true", True]

    # --- SUBTÍTULOS ---
    if (video["contentDetails"].get("caption") == "true"):
        url =  "https://www.youtube.com/watch?v=" + id_video
        # Opciones de descarga
        ydl_opts = {
            "skip_download": True,
            "writesubtitles": True,
            "writeautomaticsub": True,      # subtítulos automáticos
            "subtitleslangs": ["en"], # idioma
            "subtitlesformat": "vtt",       # formato
            "outtmpl": f"subs/{id_video}.%(ext)s",
            "quiet": True,
            "no_warnings": True
        }

        with YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])

        vtt_file = f"subs/{id_video}.en.vtt"

        if os.path.exists(vtt_file):
            with open(vtt_file, "r", encoding="utf-8") as f:
                subtitles = f.read()

            clean_sub = clean_vtt(subtitles)
            cleant_sub = clean_vtt_smart(clean_sub)

            # Borramos el archivo de subtítulos descargado tras limpiarlo
            # os.remove(vtt_file)
    
    # --- GENEROS ---
    generos = video.get("topicDetails", {}).get("topicCategories", [])

    if generos:
        generos = [
            genre.split("/")[-1].replace("_", " ")
            for genre in generos
        ]
        generos_str = ", ".join(generos)
    
    else:
        generos_str = "None"

    # --- DATAFRAME ---
    df_video = pd.DataFrame({
        "ID": id_video,
        "Titulo": video["snippet"]["title"],
        "Descripcion": video["snippet"]["description"],
        "Visualizaciones": video["statistics"]["viewCount"],
        "Numero_likes": video["statistics"].get("likeCount", None),
        "Duracion": video["contentDetails"]["duration"],
        "Fecha_publicacion": video["snippet"]["publishedAt"],
        "Titulo_canal": video["snippet"]["channelTitle"],
        "Subtitulos": cleant_sub if has_captions and cleant_sub else "None",
        "Generos": generos_str,
        "Made for kids": video["status"]["madeForKids"]
    }, index=[0])

    return df_video

In [11]:
# df_videos = []

# for id in tqdm(list_id):
#     try:
#         df_videos.append(get_info(id))
#         #print(df_videos)
#         time.sleep(0.5) #to not get too many requests error
#     except Exception as e: print("Ran into exception", e, "for video", id) #for some videos the downloader does not work for some reason

# df_data = pd.concat(df_videos, ignore_index=True)

In [12]:
# df_data

Una vez que tenemos el dataframe, lo exportamos en formato csv.

In [13]:
# data_csv = df_data.to_csv("data_videos.csv", index=False)

# **UNA SOLA FUNCIÓN PARA TODA LA EXTRACCIÓN DE DATOS**

In [14]:
def collect_all_data(num_videos):
    print("PART 1 - GETTING RANDOM IDS")
    palabras, ids = get_random_ids(num_ids=num_videos, after_date=str(datetime.date.today()-datetime.timedelta(days=1)))
    print(list(zip(palabras,ids)))

    print("PART 2 - PROCESSING VIDEOS")
    df_videos = []

    for id in tqdm(ids): #tqdm
        try:
            df_videos.append(get_info(id))
            #print(df_videos)
            time.sleep(0.2) #to not get too many requests error
        except Exception as e: pass #print("Ran into exception", e, "for video", id) #for some videos the downloader does not work for some reason

    df_data = pd.concat(df_videos, ignore_index=True)
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    data_csv = df_data.to_csv(f"data_videos_{timestamp}.csv", index=False)
    return df_data

In [ ]:
data = collect_all_data(20)#(1000)

In [ ]:
#NOTE: This function doesn't work, it produces some sort of error on part 2. It is left here as a possible idea to develop in the future.


def safe_get_info(id_video):
    try:
        return get_info(id_video)
    except Exception:
        return None

def collect_all_data_parallel(num_videos, max_workers=4):
    print("PART 1 - GETTING RANDOM IDS")
    palabras, ids = get_random_ids(
        num_ids=num_videos,
        after_date=str(datetime.date.today() - datetime.timedelta(days=1))
    )
    print(list(zip(palabras, ids)))

    print("PART 2 - PROCESSING VIDEOS")

    df_videos = []

    # Thread pool block
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(tqdm(
            executor.map(safe_get_info, ids),
            total=len(ids)
        ))

    for result in results:
        if result is not None:
            df_videos.append(result)

    if not df_videos:
        return None

    df_data = pd.concat(df_videos, ignore_index=True)

    timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    df_data.to_csv(f"data_videos_{timestamp}.csv", index=False)

    return df_data

In [ ]:
#data = collect_all_data_parallel(20)#(1000)

PART 1 - GETTING RANDOM IDS


  0%|          | 0/20 [00:00<?, ?it/s]

[('version', 'pIdKT5fGiyg'), ('hint', 'SsrF5h3CQIQ'), ('commandment', 'DRjGLkjJyOE'), ('hammock', 'gFvnChwix-k'), ('devil', 'ojARicWSqvc'), ('misogyny', '2qiKt1ZlpAk'), ('cartilage', 'wf3wSDEmYK4'), ('relaxation', 'dbRMsNKXVpI'), ('therapist', '4yv-SEX5Btc'), ('few', 'MAy3-o9Gyr0'), ('provision', 'RgLQZrUNbWw'), ('begin', 'aT_Ct-WXHrM'), ('report', 'BqB-TIAGSmw'), ('sprout', 'JTBpU5OEPRo'), ('draft', 'grXEOtW8kWU'), ('globe', 'Lb7qwifsdb4'), ('naive', 'wgljZyCo91k'), ('bomb', '4_8I0j25rRA'), ('participate', '3TDkNH8ipto'), ('attacker', 'FTkyt0stTqI')]
PART 2 - PROCESSING VIDEOS


  0%|          | 0/20 [00:00<?, ?it/s]